In [9]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV

from sklearn.metrics import (accuracy_score,f1_score,roc_auc_score,confusion_matrix,classification_report)
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
!pip install xgboost
from xgboost import XGBClassifier


from sklearn.model_selection import cross_val_score

import warnings
warnings.filterwarnings('ignore')

In [2]:
TW = pd.read_csv(
    r"../classification/Twitter/Absolute_labeling/Twitter-Absolute-Sigma-500.data",
    sep=",",
    header=None
)

groups = ["NCD", 'AI', 'AS(NA)', 'BL',
         'NAC', 'AS(NAC)', 'CS', 'AT', 'NA','ADL', 'NAD']

columns = []
for group in groups:
    for t in range(7):
        columns.append(f"{group}_{t}")

columns.append("label")  # остання колонка

TW.columns = columns

TW.head(2)

,NCD_0,NCD_1,NCD_2,NCD_3,NCD_4,NCD_5,NCD_6,AI_0,AI_1,AI_2,...,ADL_5,ADL_6,NAD_0,NAD_1,NAD_2,NAD_3,NAD_4,NAD_5,NAD_6,label
0,889,939,960,805,805,1143,1121,549,613,587,...,1.0,1.0,889,939,960,805,805,1143,1121,1.0
1,542,473,504,626,647,795,832,366,288,318,...,1.0,1.0,542,473,504,626,647,795,832,1.0


In [3]:
TH = pd.read_csv(
    "../classification/TomsHardware/Absolute_labeling/TomsHardware-Absolute-Sigma-500.data",
    sep=",",
    header=None
)


groups = [
    "NCD", "BL", "NAD", "AI", "NAC", "ND",
    "CS", "AT", "NA", "ADL", "AS_NA", "AS_NAC"
]

columns = []
for group in groups:
    for t in range(8):
        columns.append(f"{group}_{t}")

columns.append("label")  # остання колонка

TH.columns = columns


prefixes = {col.split("_")[0] for col in TH.columns if "_" in col}


TH.head()

,NCD_0,NCD_1,NCD_2,NCD_3,NCD_4,NCD_5,NCD_6,NCD_7,BL_0,BL_1,...,AS_NA_7,AS_NAC_0,AS_NAC_1,AS_NAC_2,AS_NAC_3,AS_NAC_4,AS_NAC_5,AS_NAC_6,AS_NAC_7,label
0,1,0,0,0,0,0,0,1,1.0,0.0,...,0.001816,0.001211,0.000560,0.000000,0.000000,0.000161,0.0,0.000301,0.000818,1.0
1,1,1,1,1,0,0,0,0,1.0,1.0,...,0.005029,0.000784,0.000802,0.001592,0.001612,0.000741,0.0,0.000545,0.002437,1.0
2,0,0,0,0,0,0,0,0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0
3,0,0,0,0,0,0,0,0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0
4,0,0,0,0,0,0,0,0,0.0,0.0,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.0


### Twitter

In [10]:
X = TW.drop(columns=["label"])
y = TW["label"]
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state=42,stratify=y)


### Logistic Regresion baseline

In [11]:
logisticRegr = LogisticRegression(max_iter=2000, random_state=42)
logisticRegr.fit(X_train, y_train)

pred_lr = logisticRegr.predict(X_test)
pred_proba = logisticRegr.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, pred_lr)
precision = precision_score(y_test, pred_lr)
recall = recall_score(y_test, pred_lr)
f1 = f1_score(y_test, pred_lr)
roc_auc = roc_auc_score(y_test, pred_proba)

print("Logistic Regression baseline")
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-score: {f1}")
print(f"ROC-AUC: {roc_auc}")

print("\nClassification Report:")
print(classification_report(y_test, pred_lr))

Logistic Regression baseline
Accuracy: 0.9675218534574657
Precision: 0.9399052132701422
Recall: 0.8925292529252925
F1-score: 0.9156048014773777
ROC-AUC: 0.9918065388687891

Classification Report:
              precision    recall  f1-score   support

         0.0       0.97      0.99      0.98     22587
         1.0       0.94      0.89      0.92      5555

    accuracy                           0.97     28142
   macro avg       0.96      0.94      0.95     28142
weighted avg       0.97      0.97      0.97     28142



### Logistic Regression with Scaling

In [12]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


logisticRegr = LogisticRegression(max_iter=2000, random_state=42)
logisticRegr.fit(X_train_scaled, y_train)

pred_lr = logisticRegr.predict(X_test_scaled)
pred_proba = logisticRegr.predict_proba(X_test_scaled)[:, 1]


accuracy = accuracy_score(y_test, pred_lr)
precision = precision_score(y_test, pred_lr)
recall = recall_score(y_test, pred_lr)
f1 = f1_score(y_test, pred_lr)
roc_auc = roc_auc_score(y_test, pred_proba)

print("Logistic Regression with Scaling")
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-score: {f1}")
print(f"ROC-AUC: {roc_auc}")

Logistic Regression with Scaling
Accuracy: 0.9676284556890058
Precision: 0.9409418913786555
Recall: 0.891989198919892
F1-score: 0.9158118473338878
ROC-AUC: 0.9926345085033141


### Logistic Regression with class_weight

In [13]:
logisticRegr = LogisticRegression(max_iter=2000, random_state=42, class_weight='balanced')
logisticRegr.fit(X_train, y_train)

pred_lr = logisticRegr.predict(X_test)
pred_proba = logisticRegr.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, pred_lr)
precision = precision_score(y_test, pred_lr, zero_division=0)
recall = recall_score(y_test, pred_lr, zero_division=0)
f1 = f1_score(y_test, pred_lr, zero_division=0)
roc_auc = roc_auc_score(y_test, pred_proba)

print("Logistic Regression with class_weight")
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-score: {f1}")
print(f"ROC-AUC: {roc_auc}")

Logistic Regression with class_weight
Accuracy: 0.9588870727027219
Precision: 0.8536506915406883
Recall: 0.9555355535553556
F1-score: 0.9017242843795125
ROC-AUC: 0.9922230262606551


### Logistic Regression with Scaling and class_weight

In [14]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

logisticRegr = LogisticRegression(max_iter=2000,random_state=42,class_weight='balanced')
logisticRegr.fit(X_train_scaled, y_train)

pred_lr = logisticRegr.predict(X_test_scaled)
pred_proba = logisticRegr.predict_proba(X_test_scaled)[:, 1]


accuracy = accuracy_score(y_test, pred_lr)
precision = precision_score(y_test, pred_lr, zero_division=0)
recall = recall_score(y_test, pred_lr, zero_division=0)
f1 = f1_score(y_test, pred_lr, zero_division=0)
roc_auc = roc_auc_score(y_test, pred_proba)

print("Logistic Regression with Scaling and class_weight")
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-score: {f1}")
print(f"ROC-AUC: {roc_auc}")

Logistic Regression with Scaling and class_weight
Accuracy: 0.9581053230047616
Precision: 0.8498560921010553
Recall: 0.9567956795679567
F1-score: 0.900160894233212
ROC-AUC: 0.9926690743187747


In [32]:
### Logistic regression with grsd searching

In [15]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=2000, random_state=42))
])

param_grid = {
    'model__C': [0.01, 0.1, 1, 10],
    'model__penalty': ['l2'],
    'model__class_weight': [None, 'balanced']
}

grid = GridSearchCV(pipeline,param_grid,cv=5,scoring='f1',n_jobs=-1)


grid.fit(X_train, y_train)
best_model = grid.best_estimator_

pred = best_model.predict(X_test)
accuracy = accuracy_score(y_test, pred_lr)
precision = precision_score(y_test, pred_lr, zero_division=0)
recall = recall_score(y_test, pred_lr, zero_division=0)
f1 = f1_score(y_test, pred_lr, zero_division=0)
roc_auc = roc_auc_score(y_test, pred_proba)


print('Logistic regression with grsd searching')
print("Best parameters:", grid.best_params_)
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-score: {f1}")
print(f"ROC-AUC: {roc_auc}")
print("\nClassification Report:")
print(classification_report(y_test, pred))

Logistic regression with grsd searching
Best parameters: {'model__C': 1, 'model__class_weight': None, 'model__penalty': 'l2'}
Accuracy: 0.9581053230047616
Precision: 0.8498560921010553
Recall: 0.9567956795679567
F1-score: 0.900160894233212
ROC-AUC: 0.9926690743187747

Classification Report:
              precision    recall  f1-score   support

         0.0       0.97      0.99      0.98     22587
         1.0       0.94      0.89      0.92      5555

    accuracy                           0.97     28142
   macro avg       0.96      0.94      0.95     28142
weighted avg       0.97      0.97      0.97     28142



In [21]:
import pandas as pd

data = {
    "Model": [
        "Logistic Regression",
        "Logistic Regression with Scaling",
        "Logistic Regression with class_weight",
        "Logistic Regression with Scaling and class_weight",
        "Logistic Regression with Grid Search"
    ],
    "Accuracy": [0.9675, 0.9676, 0.9589, 0.9581, 0.9581],
    "Precision ": [0.9399, 0.9409, 0.8537, 0.8499, 0.8499],
    "Recall": [0.8925, 0.8920, 0.9555, 0.9568, 0.9568],
    "F1-score": [0.9156, 0.9158, 0.9017, 0.9002, 0.9002],
    "ROC-AUC": [0.9918, 0.9926, 0.9922, 0.9927, 0.9927]
}

df = pd.DataFrame(data)
df

,Model,Accuracy,Precision,Recall,F1-score,ROC-AUC
0,Logistic Regression,0.9675,0.9399,0.8925,0.9156,0.9918
1,Logistic Regression with Scaling,0.9676,0.9409,0.8920,0.9158,0.9926
2,Logistic Regression with class_weight,0.9589,0.8537,0.9555,0.9017,0.9922
3,Logistic Regression with Scaling and class_weight,0.9581,0.8499,0.9568,0.9002,0.9927
4,Logistic Regression with Grid Search,0.9581,0.8499,0.9568,0.9002,0.9927


### Tom Hardware

In [22]:
X = TH.drop(columns=["label"])
y = TH["label"]
X_train, X_test, y_train, y_test = train_test_split(X, y,test_size=0.2,random_state=42,stratify=y)


### Logistic Regresion baseline

In [24]:
logisticRegr = LogisticRegression(max_iter=2000, random_state=42)
logisticRegr.fit(X_train, y_train)

pred_lr = logisticRegr.predict(X_test)
pred_proba = logisticRegr.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, pred_lr)
precision = precision_score(y_test, pred_lr)
recall = recall_score(y_test, pred_lr)
f1 = f1_score(y_test, pred_lr)
roc_auc = roc_auc_score(y_test, pred_proba)

print("Logistic Regression baseline")
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-score: {f1}")
print(f"ROC-AUC: {roc_auc}")

print("\nClassification Report:")
print(classification_report(y_test, pred_lr))

Logistic Regression baseline
Accuracy: 0.9607843137254902
Precision: 0.9719917012448133
Recall: 0.9639917695473251
F1-score: 0.9679752066115702
ROC-AUC: 0.9945924304161853

Classification Report:
              precision    recall  f1-score   support

         0.0       0.94      0.96      0.95       609
         1.0       0.97      0.96      0.97       972

    accuracy                           0.96      1581
   macro avg       0.96      0.96      0.96      1581
weighted avg       0.96      0.96      0.96      1581



### Logistic Regression with Scaling

In [27]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


logisticRegr = LogisticRegression(max_iter=2000, random_state=42)
logisticRegr.fit(X_train_scaled, y_train)

pred_lr = logisticRegr.predict(X_test_scaled)
pred_proba = logisticRegr.predict_proba(X_test_scaled)[:, 1]


accuracy = accuracy_score(y_test, pred_lr)
precision = precision_score(y_test, pred_lr)
recall = recall_score(y_test, pred_lr)
f1 = f1_score(y_test, pred_lr)
roc_auc = roc_auc_score(y_test, pred_proba)

print("Logistic Regression with Scaling")
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-score: {f1}")
print(f"ROC-AUC: {roc_auc}")

Logistic Regression with Scaling
Accuracy: 0.9152435167615434
Precision: 0.9665924276169265
Recall: 0.8930041152263375
F1-score: 0.9283422459893048
ROC-AUC: 0.9823210822572253


### Logistic Regression with class_weight

In [29]:
logisticRegr = LogisticRegression(max_iter=2000, random_state=42, class_weight='balanced')
logisticRegr.fit(X_train, y_train)

pred_lr = logisticRegr.predict(X_test)
pred_proba = logisticRegr.predict_proba(X_test)[:, 1]

accuracy = accuracy_score(y_test, pred_lr)
precision = precision_score(y_test, pred_lr, zero_division=0)
recall = recall_score(y_test, pred_lr, zero_division=0)
f1 = f1_score(y_test, pred_lr, zero_division=0)
roc_auc = roc_auc_score(y_test, pred_proba)

print("Logistic Regression with class_weight")
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-score: {f1}")
print(f"ROC-AUC: {roc_auc}")

Logistic Regression with class_weight
Accuracy: 0.9576217583807717
Precision: 0.9778247096092925
Recall: 0.9526748971193416
F1-score: 0.9650859822824388
ROC-AUC: 0.9945147208876455


### Logistic Regression with Scaling and class_weight

In [31]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

logisticRegr = LogisticRegression(max_iter=2000,random_state=42,class_weight='balanced')
logisticRegr.fit(X_train_scaled, y_train)

pred_lr = logisticRegr.predict(X_test_scaled)
pred_proba = logisticRegr.predict_proba(X_test_scaled)[:, 1]


accuracy = accuracy_score(y_test, pred_lr)
precision = precision_score(y_test, pred_lr, zero_division=0)
recall = recall_score(y_test, pred_lr, zero_division=0)
f1 = f1_score(y_test, pred_lr, zero_division=0)
roc_auc = roc_auc_score(y_test, pred_proba)

print("Logistic Regression with Scaling and class_weight")
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-score: {f1}")
print(f"ROC-AUC: {roc_auc}")

Logistic Regression with Scaling and class_weight
Accuracy: 0.8981657179000633
Precision: 0.9776207302709069
Recall: 0.8539094650205762
F1-score: 0.9115870400878638
ROC-AUC: 0.9825508321676903


### Logistic regression with grsd searching

In [33]:
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('model', LogisticRegression(max_iter=2000, random_state=42))
])

param_grid = {
    'model__C': [0.01, 0.1, 1, 10],
    'model__penalty': ['l2'],
    'model__class_weight': [None, 'balanced']
}

grid = GridSearchCV(pipeline,param_grid,cv=5,scoring='f1',n_jobs=-1)


grid.fit(X_train, y_train)
best_model = grid.best_estimator_

pred = best_model.predict(X_test)
accuracy = accuracy_score(y_test, pred_lr)
precision = precision_score(y_test, pred_lr, zero_division=0)
recall = recall_score(y_test, pred_lr, zero_division=0)
f1 = f1_score(y_test, pred_lr, zero_division=0)
roc_auc = roc_auc_score(y_test, pred_proba)


print('Logistic regression with grid searching')
print("Best parameters:", grid.best_params_)
print(f"Accuracy: {accuracy}")
print(f"Precision: {precision}")
print(f"Recall: {recall}")
print(f"F1-score: {f1}")
print(f"ROC-AUC: {roc_auc}")
print("\nClassification Report:")
print(classification_report(y_test, pred))

Logistic regression with grid searching
Best parameters: {'model__C': 10, 'model__class_weight': None, 'model__penalty': 'l2'}
Accuracy: 0.8981657179000633
Precision: 0.9776207302709069
Recall: 0.8539094650205762
F1-score: 0.9115870400878638
ROC-AUC: 0.9825508321676903

Classification Report:
              precision    recall  f1-score   support

         0.0       0.90      0.97      0.93       609
         1.0       0.98      0.93      0.96       972

    accuracy                           0.95      1581
   macro avg       0.94      0.95      0.94      1581
weighted avg       0.95      0.95      0.95      1581



In [35]:
data = {
    "Model": [
        "Logistic Regression",
        "Logistic Regression with Scaling",
        "Logistic Regression with class_weight",
        "Logistic Regression with Scaling and class_weight",
        "Logistic Regression with Grid Search"
    ],
    "Accuracy": [0.9608, 0.9152, 0.9576, 0.8982, 0.8982],
    "Precision (buzz)": [0.9720, 0.9666, 0.9778, 0.9776, 0.9776],
    "Recall (buzz)": [0.9640, 0.8930, 0.9527, 0.8539, 0.8539],
    "F1-score (buzz)": [0.9680, 0.9283, 0.9651, 0.9116, 0.9116],
    "ROC-AUC": [0.9946, 0.9823, 0.9945, 0.9826, 0.9826]
}

df = pd.DataFrame(data)
df

,Model,Accuracy,Precision (buzz),Recall (buzz),F1-score (buzz),ROC-AUC
0,Logistic Regression,0.9608,0.9720,0.9640,0.9680,0.9946
1,Logistic Regression with Scaling,0.9152,0.9666,0.8930,0.9283,0.9823
2,Logistic Regression with class_weight,0.9576,0.9778,0.9527,0.9651,0.9945
3,Logistic Regression with Scaling and class_weight,0.8982,0.9776,0.8539,0.9116,0.9826
4,Logistic Regression with Grid Search,0.8982,0.9776,0.8539,0.9116,0.9826
